# ➗ AI 능력 더하고 빼기 — Task Arithmetic 실험

**핵심 통찰**: 능력은 '벡터'다.
`능력 벡터(task vector) = 학습된 모델 가중치 − 원본 가중치`

이 벡터를 **더하면 능력 추가**, **빼면 능력 제거**.
레고 블록처럼 AI 능력을 끼우고 뺄 수 있다.

> 📄 논문: Ilharco et al. 2022, *Editing Models with Task Arithmetic* (arXiv:2212.04089)

**준비**: 런타임 → 런타임 유형 변경 → T4 GPU (무료)

## 1단계 — 도구 설치

In [ ]:
!pip install -q torch transformers accelerate
print('✅ 설치 완료')

## 2단계 — 능력 벡터 추출 (task vector)

`task vector = 파인튜닝된 모델 − 원본 모델` 을 가중치 단위로 계산한다.
여기선 같은 베이스의 두 모델(원본 vs 코딩 특화)로 '코딩 능력 벡터'를 뽑는다.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE = 'Qwen/Qwen2.5-1.5B'                 # 원본(pretrained)
FT   = 'Qwen/Qwen2.5-Coder-1.5B'          # 코딩 특화(finetuned) — 같은 베이스

base = AutoModelForCausalLM.from_pretrained(BASE, torch_dtype=torch.float32)
ft   = AutoModelForCausalLM.from_pretrained(FT,   torch_dtype=torch.float32)

# 🧮 능력 벡터 = ft - base  (레이어별 가중치 차이)
task_vector = {}
for (n, p_ft), (_, p_base) in zip(ft.named_parameters(), base.named_parameters()):
    task_vector[n] = (p_ft.data - p_base.data)
print('✅ 코딩 능력 벡터 추출 완료 —', len(task_vector), '개 텐서')

## 3단계 — 능력 더하기 (+) 🔧

원본 모델에 능력 벡터를 `scaling`만큼 더해 코딩 능력을 '이식'한다.
`scaling`을 키우면 더 강하게(너무 크면 망가짐), 0.5~1.0이 보통.

In [ ]:
import copy

def apply_task_vector(base_model, tv, scaling=1.0):
    m = copy.deepcopy(base_model)
    for n, p in m.named_parameters():
        if n in tv:
            p.data = p.data + scaling * tv[n]   # ➕ 능력 더하기 (빼려면 -scaling)
    return m

added = apply_task_vector(base, task_vector, scaling=1.0)   # 코딩 능력 +
added = added.to('cuda' if torch.cuda.is_available() else 'cpu').to(torch.bfloat16)
tok = AutoTokenizer.from_pretrained(BASE)
print('✅ 원본 + 코딩능력벡터 = 코딩 가능한 모델 탄생')

## 4단계 — 테스트 (능력이 이식됐나)

In [ ]:
def ask(model, q):
    msgs = [{'role':'user','content':q}]
    inputs = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(model.device)
    out = model.generate(inputs, max_new_tokens=160, do_sample=False)
    print(tok.decode(out[0][inputs.shape[-1]:], skip_special_tokens=True))

print('=== 능력 더한 모델 (코딩 질문) ===')
ask(added, '파이썬으로 리스트를 정렬하는 함수를 써줘')

## 5단계 (실험) — 능력 빼기 (−) 와 강도 조절

- `scaling = -1.0` → 코딩 능력을 **빼서** 일부러 못하게 (forgetting)
- `scaling = 0.3 / 0.7 / 1.5` → 능력 강도가 어떻게 변하는지 비교

In [ ]:
for s in [0.3, 0.7, 1.0]:
    print(f'\n===== scaling = {s} =====')
    m = apply_task_vector(base, task_vector, scaling=s).to('cuda' if torch.cuda.is_available() else 'cpu').to(torch.bfloat16)
    ask(m, '파이썬으로 피보나치 함수 써줘')
    del m; torch.cuda.empty_cache()

---
## 🎓 무슨 일이 일어난 건가

- **능력을 벡터로 다뤘다**: `코딩능력 = 코딩모델 − 원본`
- 그 벡터를 원본에 **더하니(+)** 원본이 코딩을 하게 됐다 (학습 없이!)
- **빼면(−)** 능력이 사라지고, **강도(scaling)**로 세기를 조절한다
- 이게 model merging·abliteration의 수학적 토대다 (전부 '벡터 산술')

**다음 실험:**
- 두 능력 벡터를 동시에 더하기 (`코딩 + 수학`)
- 한 능력은 더하고 다른 건 빼기
- 여러 task vector를 평균 (= model merging 과 연결)

> 🔬 이건 안전하고 건설적인 weight 수술 — 능력을 **더하는** 방향. (검열 제거 abliteration과 다름)